In [1]:
import pandas as pd
import scanpy as sc
import numpy as np
import glob 
from anndata import AnnData
from scipy.sparse import csr_matrix
import h5py


/home/proth/miniconda3/envs/TOSICA/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
lineage_info_df = pd.read_csv('/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/morris/b4.rna&atac.r1&2_master_rev1_annotated.csv')
lineage_info_df["barcode_stripped"] = [f[16:] for f in lineage_info_df["cell.bc"]] #for ATAC delete first 16 characters
lineage_info_df = lineage_info_df.set_index("barcode_stripped")
# Keep only the first occurrence of each duplicate index
lineage_info_df = lineage_info_df.loc[~lineage_info_df.index.duplicated(keep='first')]
cells_with_lineage_info = lineage_info_df.index

In [3]:
filepath = "/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/raw_datasets/morris/GSM6705332_B4D21-ATAC-r2-1_filtered_peak_bc_matrix.h5"

In [4]:
# make empty anndata object


with h5py.File(filepath, 'r') as f:
# Inspect the contents of the "matrix" group


# Extract the datasets
    data = f['matrix/data'][:]
    indices = f['matrix/indices'][:]
    indptr = f['matrix/indptr'][:]
    shape = f['matrix/shape'][:]

    # Extract barcodes
    barcodes = f['matrix/barcodes'][:]


    # Extract feature names
    features = f['matrix/features/name'][:]



shape = (shape[1], shape[0])

# Convert to CSR matrix
matrix = csr_matrix((data, indices, indptr), shape=shape)

# Decode barcodes and features if they are in bytes
barcodes = [barcode.decode('utf-8') for barcode in barcodes]
features = [feature.decode('utf-8') for feature in features]

# Create AnnData object
adata = AnnData(X=matrix, obs=pd.DataFrame(index=barcodes), var=pd.DataFrame(index=features))

cells_with_lineage_info_in_adata1 = [f for f in adata.obs_names if f in cells_with_lineage_info.values]
adata = adata[cells_with_lineage_info_in_adata1]


In [5]:
adata

View of AnnData object with n_obs × n_vars = 1892 × 163646

In [6]:
print(adata.shape)
adata.obs["barcode"] = adata.obs_names.map(lineage_info_df["clone_id"])


(1892, 163646)


/tmp/ipykernel_3259879/3380203701.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["barcode"] = adata.obs_names.map(lineage_info_df["clone_id"])


In [7]:
adata.var_names

Index(['chr1:3119540-3120453', 'chr1:3267991-3268890', 'chr1:3360536-3361569',
       'chr1:3416004-3416747', 'chr1:3444651-3445528', 'chr1:3477066-3477987',
       'chr1:3493387-3494279', 'chr1:3514577-3515495', 'chr1:3516716-3517582',
       'chr1:3552692-3553506',
       ...
       'GL456216.1:13425-14333', 'GL456216.1:15634-16538',
       'GL456216.1:16726-17487', 'GL456216.1:30576-31473',
       'GL456216.1:34653-35589', 'GL456216.1:40644-41196',
       'GL456216.1:41293-41683', 'GL456216.1:48804-49711',
       'JH584292.1:12570-13470', 'JH584295.1:1566-1976'],
      dtype='object', length=163646)

In [8]:
df = pd.DataFrame(adata.X.todense(), columns = adata.var_names)
df["barcode"] = adata.obs["barcode"].values
df["barcode_cont"] = pd.factorize(df["barcode"])[0]
df = df.drop(columns = ["barcode"])

In [9]:
df

,chr1:3119540-3120453,chr1:3267991-3268890,chr1:3360536-3361569,chr1:3416004-3416747,chr1:3444651-3445528,chr1:3477066-3477987,chr1:3493387-3494279,chr1:3514577-3515495,chr1:3516716-3517582,chr1:3552692-3553506,...,GL456216.1:15634-16538,GL456216.1:16726-17487,GL456216.1:30576-31473,GL456216.1:34653-35589,GL456216.1:40644-41196,GL456216.1:41293-41683,GL456216.1:48804-49711,JH584292.1:12570-13470,JH584295.1:1566-1976,barcode_cont
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,1,2,0,0,0,0,0,0,0,1
2,0,0,0,1,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,2
3,0,0,0,0,0,0,0,0,0,0,...,3,5,0,0,0,0,0,0,0,3
4,0,0,0,0,0,0,0,0,0,0,...,1,2,0,0,0,0,0,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1887,0,0,0,0,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,29
1888,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,319
1889,0,0,0,2,0,0,0,2,0,0,...,1,1,0,0,0,0,0,0,0,802
1890,0,0,0,0,0,0,0,0,0,0,...,0,2,0,0,0,0,0,0,0,89


In [10]:
df.to_pickle("/fast/AG_Metzger/philipp/python_files/gene_signature/mutual_information_approach/November_25_ReDo/preprocessing/pkl_files_filtered/morris_ATAC.pkl")

